In [12]:
from pathlib import Path
from sklearn.experimental import enable_iterative_imputer  # noqa
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb
import xgboost as xgb
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import warnings
import pickle, copy
warnings.filterwarnings('ignore')


In [13]:
DATA_DIR = Path(r".")

df_P5_flow = pd.read_excel(
    DATA_DIR / "origin3.xlsx",
    usecols=["Data Time", "T", "V", "Q"]
)

df_P5_flow["Data Time"] = pd.to_datetime(df_P5_flow["Data Time"]).dt.floor("h")
df_P5_flow = df_P5_flow.rename(columns={
    "Data Time": "datetime", "T": "T_water", "V": "v", "Q": "Q"
})

df_P5_flow = df_P5_flow.drop_duplicates(subset="datetime")
df_P5_flow = df_P5_flow.set_index("datetime").sort_index()

df_rain = pd.read_excel(
    DATA_DIR / "rain.xlsx",
    usecols=["Data Time", "Rainfall (mm)", "Air temperature"]
)
df_rain["Data Time"] = pd.to_datetime(df_rain["Data Time"])
df_rain = df_rain.rename(columns={
    "Data Time": "datetime", "Rainfall (mm)": "rain_mm", "Air temperature": "T_air"
})
df_rain = df_rain.drop_duplicates(subset="datetime")
df_rain = df_rain.set_index("datetime").sort_index()

In [14]:
def find_complete_segments(df, cols=['T_water', 'v', 'Q']):
    mask = df[cols].notna().all(axis=1)
    
    segments = []
    start = None
    length = 0
    
    for idx, is_complete in mask.items():
        if is_complete:
            if start is None:
                start = idx
            length += 1
        else:
            if start is not None:
                end = df.index[df.index.get_loc(idx) - 1]
                segments.append({'start': start, 'end': end, 'length': length})
                start = None
                length = 0
    if start is not None:
        segments.append({'start': start, 'end': df.index[-1], 'length': length})
    
    return pd.DataFrame(segments)

complete_segs = find_complete_segments(df_P5_flow, cols=['T_water', 'v', 'Q'])

In [15]:
def create_missing_mask(
    df_orig,
    cols_to_mask=['Q', 'v'],
    total_missing_ratio=0.2,
    block_ratio=0.8,
    block_length_range=(2, 48),
    mode_ratio={'Q_only': 0.2, 'Qv_both': 0.8},
    random_state=42
):
    np.random.seed(random_state)
    df_masked = df_orig.copy()
    mask_df = pd.DataFrame(False, index=df_orig.index, columns=cols_to_mask)

    n_rows = len(df_orig)
    total_missing_points = int(n_rows * total_missing_ratio)
    if total_missing_points == 0:
        return df_masked, mask_df

    q_ok = df_orig['Q'].notna().to_numpy()
    v_ok = df_orig['v'].notna().to_numpy()

    n_block_missing = int(total_missing_points * block_ratio)
    n_random_missing = total_missing_points - n_block_missing

    indices = df_orig.index.to_list()
    used_mask = np.zeros(n_rows, dtype=bool)

    current_block_missing = 0
    attempts = 0
    max_attempts = 200

    while current_block_missing < n_block_missing and attempts < max_attempts:
        max_start = n_rows - block_length_range[0]
        if max_start <= 0:
            break
        start_pos = np.random.randint(0, max_start)
        block_len = np.random.randint(block_length_range[0], block_length_range[1] + 1)
        end_pos = min(start_pos + block_len, n_rows)

        block_positions = range(start_pos, end_pos)
        if any(used_mask[i] for i in block_positions):
            attempts += 1
            continue
        if not all(q_ok[i] for i in block_positions):
            attempts += 1
            continue

        block_size = end_pos - start_pos
        n_q_only = int(block_size * mode_ratio['Q_only'])
        n_both = block_size - n_q_only

        modes = ['Q_only'] * n_q_only + ['Qv_both'] * n_both
        np.random.shuffle(modes)

        for offset, mode in enumerate(modes):
            row_idx = start_pos + offset
            if used_mask[row_idx]:
                continue
            used_mask[row_idx] = True
            current_block_missing += 1
            time_point = indices[row_idx]
            if mode == 'Q_only':
                df_masked.loc[time_point, 'Q'] = np.nan
                mask_df.loc[time_point, 'Q'] = True
            else:  # Qv_both
                df_masked.loc[time_point, 'Q'] = np.nan
                mask_df.loc[time_point, 'Q'] = True
                if v_ok[row_idx]:
                    df_masked.loc[time_point, 'v'] = np.nan
                    mask_df.loc[time_point, 'v'] = True
        attempts += 1

    available_rows = [i for i in range(n_rows) if not used_mask[i] and q_ok[i]]
    n_random = min(n_random_missing, len(available_rows))
    if n_random > 0:
        selected = np.random.choice(available_rows, size=n_random, replace=False)

        n_q_only_random = int(n_random * mode_ratio['Q_only'])
        n_both_random = n_random - n_q_only_random
        modes_random = ['Q_only'] * n_q_only_random + ['Qv_both'] * n_both_random
        np.random.shuffle(modes_random)
        for idx, mode in zip(selected, modes_random):
            time_point = indices[idx]
            if mode == 'Q_only':
                df_masked.loc[time_point, 'Q'] = np.nan
                mask_df.loc[time_point, 'Q'] = True
            else:
                df_masked.loc[time_point, 'Q'] = np.nan
                mask_df.loc[time_point, 'Q'] = True
                if v_ok[idx]:
                    df_masked.loc[time_point, 'v'] = np.nan
                    mask_df.loc[time_point, 'v'] = True

    return df_masked, mask_df


# ==================== 2. 数据合并与预处理 ====================
# 合并流量与降雨/气温数据（只写一次）
df_P5 = df_P5_flow.join(df_rain[['rain_mm', 'T_air']], how='left')

# 补全为连续小时网格
full_grid = pd.date_range(df_P5.index.min(), df_P5.index.max(), freq='h')
df_P5 = df_P5.reindex(full_grid)
df_P5.index.name = 'datetime'

# 数据检查（可选）
dup = df_P5.index.duplicated().sum()
time_diffs = pd.Series(df_P5.index).diff().dropna()

rain_nan = df_P5['rain_mm'].isna().sum()
tair_nan = df_P5['T_air'].isna().sum()
if rain_nan == 0 and tair_nan == 0:
    print("\n Aligned")
else:
    print("\n Misaligned")

for col in ['Q', 'v', 'T_water', 'rain_mm', 'T_air']:
    print(f"  {col:10s}: min={df_P5[col].min():.3f}  "
          f"max={df_P5[col].max():.3f}  "
          f"mean={df_P5[col].mean():.3f}")


# ==================== 3. 特征工程 ====================
def build_features(df):
    d = df.copy()

    for lag in [1, 2, 3, 6, 9, 12, 18, 24]:
        d[f'rain_lag{lag}'] = d['rain_mm'].shift(lag)

    for win in [6, 12, 24]:
        d[f'rain_sum{win}'] = d['rain_mm'].shift(1).rolling(win).sum()

    for lag in [1, 2, 3, 6, 12, 24]:
        d[f'Q_lag{lag}'] = d['Q'].shift(lag)

    for lag in [1, 2, 3, 6]:
        d[f'v_lag{lag}'] = d['v'].shift(lag)

    d['hour']      = d.index.hour
    d['dayofyear'] = d.index.dayofyear
    d['hour_sin']  = np.sin(2 * np.pi * d['hour'] / 24)
    d['hour_cos']  = np.cos(2 * np.pi * d['hour'] / 24)
    d['doy_sin']   = np.sin(2 * np.pi * d['dayofyear'] / 365)
    d['doy_cos']   = np.cos(2 * np.pi * d['dayofyear'] / 365)

    return d

df_P5_feat = build_features(df_P5)

# 去掉非 Q/v/T_water 列中的 NaN（滞后特征、降雨等产生的缺失）
df_P5_feat = df_P5_feat.dropna(
    subset=[c for c in df_P5_feat.columns if c not in ['Q', 'v', 'T_water']]
)

# 只保留 Q/v/T_water 齐全的行，作为完整参考
complete_df = df_P5_feat.dropna(subset=['Q', 'v', 'T_water']).copy()


# ==================== 4. 划分数据集（首尾相接，不重不漏） ====================
train_orig = complete_df.loc['2024-01-01 00:00:00':'2024-10-31 04:00:00']
val_orig   = complete_df.loc['2024-10-31 05:00:00':'2024-11-20 02:00:00'] 
test_orig  = complete_df.loc['2024-11-20 03:00:00':'2024-12-31 23:00:00']


# ==================== 5. 生成缺失掩码 ====================
train_masked, train_mask = create_missing_mask(train_orig, total_missing_ratio=0.2, random_state=42)
val_masked,   val_mask   = create_missing_mask(val_orig,   total_missing_ratio=0.2, random_state=43)
test_masked,  test_mask  = create_missing_mask(test_orig,  total_missing_ratio=0.2, random_state=44)


# ==================== 6. 导出结果 ====================
train_masked.to_excel("train_masked.xlsx")
val_masked.to_excel("val_masked.xlsx")
test_masked.to_excel("test_masked.xlsx")

train_mask.to_excel("train_mask.xlsx")
val_mask.to_excel("val_mask.xlsx")
test_mask.to_excel("test_mask.xlsx")

train_masked, val_masked, test_masked


 Aligned
  Q         : min=0.151  max=103.468  mean=2.861
  v         : min=0.059  max=1.265  mean=0.362
  T_water   : min=11.100  max=34.800  mean=21.997
  rain_mm   : min=0.000  max=17.000  mean=0.145
  T_air     : min=2.050  max=41.600  mean=19.843


(                     T_water      v         Q  rain_mm      T_air  rain_lag1  \
 datetime                                                                       
 2024-01-02 00:00:00     12.3  0.361  1.328017      0.0   9.300000        0.0   
 2024-01-02 01:00:00     12.3  0.385  1.353051      0.0   9.300000        0.0   
 2024-01-02 02:00:00     12.3  0.368  1.323374      0.0   9.300000        0.0   
 2024-01-02 03:00:00     12.3  0.355  1.188611      0.0   9.300000        0.0   
 2024-01-02 04:00:00     12.3  0.343  1.107984      0.0   9.300000        0.0   
 ...                      ...    ...       ...      ...        ...        ...   
 2024-10-30 16:00:00     22.3  0.335  1.004165      0.0  23.400000        0.0   
 2024-10-30 17:00:00     22.3  0.339  0.968662      0.0  23.450001        0.0   
 2024-10-30 22:00:00     22.0  0.333  0.921111      0.0  20.200001        0.0   
 2024-10-30 23:00:00     22.0  0.336  0.896111      0.0  19.500000        0.0   
 2024-10-31 01:00:00     22.

In [16]:
V_FEATURES = (
    ['T_water', 'T_air', 'rain_mm'] +
    [f'rain_lag{l}' for l in [1,2,3,6,9,12,18,24]] +
    [f'rain_sum{w}' for w in [6,12,24]] +
    [f'v_lag{l}' for l in [1,2,3,6]] +
    ['hour_sin', 'hour_cos', 'doy_sin', 'doy_cos']
)

Q_FEATURES = (
    ['T_water', 'T_air', 'rain_mm', 'v'] +
    [f'rain_lag{l}' for l in [1,2,3,6,9,12,18,24]] +
    [f'rain_sum{w}' for w in [6,12,24]] +
    [f'Q_lag{l}' for l in [1,2,3,6,12,24]] +
    [f'v_lag{l}' for l in [1,2,3,6]] +
    ['hour_sin', 'hour_cos', 'doy_sin', 'doy_cos']
)

In [17]:
def build_features_lstm(df):
    d = pd.DataFrame(index=df.index)
    d['Q'], d['v'] = df['Q'], df['v']
    d['T_water'], d['T_air'] = df['T_water'], df['T_air']

    rain = df['rain_mm'].fillna(0.0)
    d['rain_mm'] = rain
    for l in (1, 2, 3, 6, 9, 12, 18, 24):
        d[f'rain_lag{l}'] = rain.shift(l)
    for w in (6, 12, 24):
        d[f'rain_sum{w}'] = rain.rolling(w, min_periods=1).sum()

    logQ = np.log(df['Q'].clip(lower=1e-3))
    logv = np.log(df['v'].clip(lower=1e-3))
    A = logQ - logv
    d['logQ_last'], d['Q_age'] = _last_and_age(logQ)
    d['logv_last'], d['v_age'] = _last_and_age(logv)
    d['logA_last'], d['A_age'] = _last_and_age(A)
    d['logQ_k24'] = _lastk(logQ, 24, 'mean')
    d['logQ_k72'] = _lastk(logQ, 72, 'mean')
    d['logv_k24'] = _lastk(logv, 24, 'mean')
    d['logv_k72'] = _lastk(logv, 72, 'mean')
    d['Q_age'] = d['Q_age'].clip(upper=AGE_CAP)
    d['v_age'] = d['v_age'].clip(upper=AGE_CAP)
    d['A_age'] = d['A_age'].clip(upper=AGE_CAP)

    d['hour_sin'] = np.sin(2 * np.pi * df.index.hour / 24)
    d['hour_cos'] = np.cos(2 * np.pi * df.index.hour / 24)
    d['doy_sin'] = np.sin(2 * np.pi * df.index.dayofyear / 365.25)
    d['doy_cos'] = np.cos(2 * np.pi * df.index.dayofyear / 365.25)
    return d

LOG_COLS = ['Q']
LSTM_FEATURES = (
    ['Q', 'v', 'T_water', 'T_air', 'rain_mm'] +
    [f'rain_lag{l}' for l in [1, 2, 3, 6, 9, 12, 18, 24]] +
    [f'rain_sum{w}' for w in [6, 12, 24]] +
    ['logQ_last', 'Q_age', 'logv_last', 'v_age',
     'logA_last', 'A_age',
     'logQ_k24', 'logQ_k72', 'logv_k24', 'logv_k72'] +
    ['hour_sin', 'hour_cos', 'doy_sin', 'doy_cos']
)
assert len(LSTM_FEATURES) == 30

class DataScaler:
    def __init__(self, feature_cols):
        self.feature_cols = feature_cols
        self.scalers = {}

    def fit(self, df):
        for col in self.feature_cols:
            vals = df[col].dropna().values.reshape(-1, 1)
            sc = StandardScaler()
            if col in LOG_COLS:
                sc.fit(np.log1p(np.clip(vals, 0, None)))
            else:
                sc.fit(vals)
            self.scalers[col] = (sc, col in LOG_COLS)

    def transform_array(self, df):
        out = np.full((len(df), len(self.feature_cols)), np.nan)
        for i, col in enumerate(self.feature_cols):
            sc, use_log = self.scalers[col]
            vals = df[col].values.reshape(-1, 1).astype(float)
            notna = ~np.isnan(vals.ravel())
            if notna.any():
                v = vals[notna].reshape(-1, 1)
                if use_log:
                    v = np.log1p(np.clip(v, 0, None))
                out[notna, i] = sc.transform(v).ravel()
        return out

    def inverse_Q(self, scaled_vals):
        sc, _ = self.scalers['Q']
        log_v = sc.inverse_transform(np.array(scaled_vals).reshape(-1, 1))
        return np.expm1(log_v).ravel()

    def inverse_v(self, scaled_vals):
        sc, _ = self.scalers['v']
        return sc.inverse_transform(np.array(scaled_vals).reshape(-1, 1)).ravel()

def train_lstm(train_masked, train_orig, scaler,
               window= 96, epochs=30, batch_size=64, lr=1e-3, device='cpu'):

    train_scaled      = scaler.transform_array(train_masked)
    train_orig_scaled = scaler.transform_array(train_orig)
    missing_mask      = np.isnan(train_scaled[:, :2])

    n_features   = train_scaled.shape[1]  
    input_size   = n_features + 2

    dataset = ImputationDataset(train_scaled, train_orig_scaled,
                                missing_mask, window)
    loader  = DataLoader(dataset, batch_size=batch_size,
                         shuffle=True, num_workers=0)

    model = LSTMImputer(
        input_size=input_size,
        hidden_size=128,
        num_layers=2
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=5, factor=0.5)

    model.train()
    for epoch in range(epochs):
        total_loss, n_batch = 0.0, 0
        for x, y, m in loader:
            x, y, m = x.to(device), y.to(device), m.to(device)
            if not m.any():
                continue
            pred      = model(x)
            loss_elem = (pred - y) ** 2
            weights   = m.float()
            loss = (loss_elem * weights).sum() / (weights.sum() + 1e-8)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
            n_batch    += 1

        avg_loss = total_loss / max(n_batch, 1)
        scheduler.step(avg_loss)
        if (epoch + 1) % 10 == 0:
            print(f"    Epoch {epoch+1}/{epochs}  Loss: {avg_loss:.4f}  "
                  f"LR: {optimizer.param_groups[0]['lr']:.2e}")

    return model

In [18]:
def kge(true, pred):
    r     = np.corrcoef(true, pred)[0, 1]
    alpha = np.std(pred) / (np.std(true) + 1e-8)
    beta  = np.mean(pred) / (np.mean(true) + 1e-8)
    return 1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)

def smape(true, pred):
    return np.mean(2 * np.abs(true - pred) /
                   (np.abs(true) + np.abs(pred) + 1e-8)) * 100

def mae(true, pred):
    return np.mean(np.abs(true - pred))

def evaluate_imputation(orig_df, filled_df, mask_df, col, method_name, split_name):
    idx = mask_df[col]
    if idx.sum() == 0:
        return {}

    true_vals   = orig_df.loc[idx, col].values
    filled_vals = filled_df.loc[idx, col].values

    valid       = ~np.isnan(filled_vals)
    true_vals   = true_vals[valid]
    filled_vals = filled_vals[valid]

    if len(true_vals) == 0:
        return {}

    return {
        'method'   : method_name,
        'split'    : split_name,
        'col'      : col,
        'n_imputed': int(valid.sum()),
        'KGE'      : round(kge(true_vals, filled_vals), 4),
        'MAE'      : round(mae(true_vals, filled_vals), 4),
        'SMAPE(%)'  : round(smape(true_vals, filled_vals), 4),
    }

In [19]:
class LSTMImputer(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2, output_size=2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size, num_layers,
            batch_first=True, bidirectional=False, dropout=0.2)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out)

class ImputationDataset(Dataset):
    def __init__(self, scaled_arr, orig_scaled_arr, missing_mask, window= 96):
        self.samples = []
        n, F = scaled_arr.shape
        x_filled   = np.where(np.isnan(scaled_arr), 0.0, scaled_arr)
        is_missing = missing_mask.astype(np.float32)
        x_input    = np.concatenate([x_filled, is_missing], axis=1)

        for i in range(n - window + 1):
            x_seq = x_input[i:i+window]
            y_seq = orig_scaled_arr[i:i+window, :2]  
            m_seq = missing_mask[i:i+window]
            self.samples.append((x_seq, y_seq, m_seq))

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        x, y, m = self.samples[idx]
        return torch.FloatTensor(x), torch.FloatTensor(y), torch.BoolTensor(m)

def impute_lstm(masked_df, model, scaler, window= 96, device='cpu'):
    scaled     = scaler.transform_array(masked_df)
    x_filled   = np.where(np.isnan(scaled), 0.0, scaled)
    is_missing = np.isnan(scaled[:, :2]).astype(np.float32)
    x_input    = np.concatenate([x_filled, is_missing], axis=1)

    n = len(x_input)
    pred_sum   = np.zeros((n, 2))
    pred_count = np.zeros((n, 2))

    model.eval()
    with torch.no_grad():
        for i in range(max(1, n - window + 1)):
            end   = min(i + window, n)
            x_seq = torch.FloatTensor(x_input[i:end]).unsqueeze(0).to(device)
            pred  = model(x_seq).squeeze(0).cpu().numpy()
            pred_sum[i:end]   += pred
            pred_count[i:end] += 1

    pred_avg  = pred_sum / np.maximum(pred_count, 1)
    filled_df = masked_df.copy()

    q_mask = masked_df['Q'].isna()
    if q_mask.any():
        filled_df.loc[q_mask, 'Q'] = scaler.inverse_Q(pred_avg[q_mask.values, 0])
    v_mask = masked_df['v'].isna()
    if v_mask.any():
        filled_df.loc[v_mask, 'v'] = scaler.inverse_v(pred_avg[v_mask.values, 1])

    return filled_df

In [20]:
def _last_and_age(s):
    pos = np.arange(len(s), dtype=float)
    last_pos = pd.Series(np.where(s.notna().to_numpy(), pos, np.nan),
                         index=s.index).ffill().shift(1)
    return s.ffill().shift(1), pd.Series(pos, index=s.index) - last_pos

def _lastk(s, k, how='mean'):
    r = s.dropna().rolling(k, min_periods=1)
    return (r.median() if how == 'median' else r.mean()).reindex(s.index).ffill().shift(1)

AGE_CAP = 720 

def build_features_v2(df):

    d = build_features(df)

    q_ffill = df['Q'].ffill().bfill()
    v_ffill = df['v'].ffill().bfill()

    logQ = np.log(df['Q'].clip(lower=1e-3))
    logv = np.log(df['v'].clip(lower=1e-3))
    A = logQ - logv
    d['logQ_last'], d['Q_age'] = _last_and_age(logQ)
    d['logv_last'], d['v_age'] = _last_and_age(logv)
    d['logA_last'], d['A_age'] = _last_and_age(A)
    d['logQ_k24'] = _lastk(logQ, 24, 'mean')
    d['logQ_k72'] = _lastk(logQ, 72, 'mean')
    d['logv_k24'] = _lastk(logv, 24, 'mean')
    d['logv_k72'] = _lastk(logv, 72, 'mean')
    d['Q_age'] = d['Q_age'].clip(upper=AGE_CAP)
    d['v_age'] = d['v_age'].clip(upper=AGE_CAP)
    d['A_age'] = d['A_age'].clip(upper=AGE_CAP)
    
    for win in [24, 72, 168]:
        rm = q_ffill.rolling(win, min_periods=1).mean()
        rs = q_ffill.rolling(win, min_periods=1).std().fillna(1e-8)
        d[f'Q_obs_mean{win}']   = rm
        d[f'Q_obs_std{win}']    = rs
        d[f'Q_obs_zscore{win}'] = (q_ffill - rm) / (rs + 1e-8)
        d[f'Q_obs_max{win}']    = q_ffill.rolling(win, min_periods=1).max()

    for win in [24, 72]:
        rm = v_ffill.rolling(win, min_periods=1).mean()
        rs = v_ffill.rolling(win, min_periods=1).std().fillna(1e-8)
        d[f'v_obs_mean{win}']   = rm
        d[f'v_obs_zscore{win}'] = (v_ffill - rm) / (rs + 1e-8)

    for h in [6, 24, 72]:
        d[f'Q_trend{h}'] = q_ffill - q_ffill.shift(h)
        d[f'v_trend{h}'] = v_ffill - v_ffill.shift(h)

    d['is_flood_season'] = d.index.month.isin([5, 6, 7, 8, 9]).astype(float)
    d['month_sin'] = np.sin(2 * np.pi * d.index.month / 12)
    d['month_cos'] = np.cos(2 * np.pi * d.index.month / 12)

    return d


def evaluate_all(orig_df, filled_df, mask_df, method_name, split_name):
    results = []
    for col in ['Q', 'v']:
        r = evaluate_imputation(orig_df, filled_df, mask_df,
                                col, method_name, split_name)
        if r:
            results.append(r)
    return results

def finetune_on_target(model, target_masked, target_orig, scaler,
                       window= 96, epochs=10, lr=1e-4, device='cpu',
                       watch_frac=0.1, seed=10):
    torch.manual_seed(seed)
    model_ft = copy.deepcopy(model)


    for name, p in model_ft.named_parameters():
        if '_l0' in name:
            p.requires_grad_(False)
    optimizer = torch.optim.Adam(
        [p for p in model_ft.parameters() if p.requires_grad], lr=lr)

    target_scaled      = scaler.transform_array(target_masked)
    target_orig_scaled = scaler.transform_array(target_orig)

    obs_mask = ~np.isnan(target_scaled[:, :2])

    x_filled = np.where(np.isnan(target_scaled), 0.0, target_scaled)

    x_filled[:, 0] = 0.0       
    x_filled[:, 1] = 0.0
    is_missing = np.ones((len(x_filled), 2), dtype=np.float32)
    x_input = np.concatenate([x_filled, is_missing], axis=1)

    y = np.where(np.isnan(target_orig_scaled), 0.0, target_orig_scaled)

    n = len(x_input)
    n_fit = n - max(1, int(n * watch_frac))
    all_starts   = list(range(0, max(1, n - window + 1)))
    fit_starts   = [i for i in all_starts if i <  max(1, n_fit - window + 1)]
    watch_starts = [i for i in all_starts if i >= max(1, n_fit - window + 1)]
    if not fit_starts:
        fit_starts = all_starts[:1]

    def _obs_loss(starts, do_step):
        tot, cnt = 0.0, 0
        for i in starts:
            end   = min(i + window, n)
            x_seq = torch.FloatTensor(x_input[i:end]).unsqueeze(0).to(device)
            y_seq = torch.FloatTensor(y[i:end, :2]).unsqueeze(0).to(device)
            m_seq = torch.BoolTensor(obs_mask[i:end]).unsqueeze(0).to(device)
            pred  = model_ft(x_seq)
            l = ((pred - y_seq) ** 2 * m_seq.float()).sum() / (m_seq.float().sum() + 1e-8)
            if do_step:
                optimizer.zero_grad()
                l.backward()
                torch.nn.utils.clip_grad_norm_(
                    [p for p in model_ft.parameters() if p.requires_grad], 1.0)
                optimizer.step()
            tot += l.item()
            cnt += 1
        return tot / max(cnt, 1)

    best_loss, best_state = np.inf, None
    model_ft.train()
    for epoch in range(epochs):
        fit_loss = _obs_loss(fit_starts, do_step=True)
        model_ft.eval()
        with torch.no_grad():
            watch_loss = _obs_loss(watch_starts, do_step=False) if watch_starts else fit_loss
        model_ft.train()
        if watch_loss < best_loss:
            best_loss, best_state = watch_loss, copy.deepcopy(model_ft.state_dict())

        if (epoch + 1) % 5 == 0:
            print(f"      FT Epoch {epoch+1}/{epochs}  "
                  f"Loss: {fit_loss:.6f}  watch: {watch_loss:.6f}")

    if best_state is not None:
        model_ft.load_state_dict(best_state)
    model_ft.eval()
    return model_ft

def run_comparison(device='cpu'):

    all_results = []

    print ("Build feature dataset ..")
    df_P5_v2 = build_features_v2(df_P5)
    df_P5_v2 = df_P5_v2.dropna(subset=[c for c in df_P5_v2.columns
                                       if c not in ['Q', 'v', 'T_water']])
    complete_v2   = df_P5_v2.dropna(subset=['Q', 'v', 'T_water']).copy()
    train_orig_v2 = complete_v2.loc['2024-01-02 00:00:00':'2024-10-31 04:00:00']
    val_orig_v2   = complete_v2.loc['2024-10-31 05:00:00':'2024-11-20 02:00:00']
    test_orig_v2  = complete_v2.loc['2024-11-20 03:00:00':'2024-12-29 23:00:00']

    train_masked_v2, train_mask_v2 = create_missing_mask(train_orig_v2)
    val_masked_v2,   val_mask_v2   = create_missing_mask(val_orig_v2)
    test_masked_v2,  test_mask_v2  = create_missing_mask(test_orig_v2)

    LSTM_FEATURES_V2 = (
        ['Q', 'v', 'T_water', 'T_air', 'rain_mm'] +
        [f'rain_lag{l}'     for l in [1, 2, 3, 6, 9, 12, 18, 24]] +
        [f'rain_sum{w}'     for w in [6, 12, 24]] +
        ['logQ_last', 'Q_age', 'logv_last', 'v_age',
         'logA_last', 'A_age',
         'logQ_k24', 'logQ_k72',
         'logv_k24', 'logv_k72'] +
        [f'Q_obs_mean{w}'   for w in [24, 72, 168]] +
        [f'Q_obs_std{w}'    for w in [24, 72, 168]] +
        [f'Q_obs_zscore{w}' for w in [24, 72, 168]] +
        [f'Q_obs_max{w}'    for w in [24, 72, 168]] +
        [f'v_obs_mean{w}'   for w in [24, 72]] +
        [f'v_obs_zscore{w}' for w in [24, 72]] +
        [f'Q_trend{h}'      for h in [6, 24, 72]] +
        [f'v_trend{h}'      for h in [6, 24, 72]] +
        ['hour_sin', 'hour_cos', 'doy_sin', 'doy_cos',
         'month_sin', 'month_cos', 'is_flood_season']
    )
    assert len(LSTM_FEATURES_V2) == 55

    missing_cols = [c for c in LSTM_FEATURES_V2 if c not in df_P5_v2.columns]
    assert not missing_cols, f"Build_features-v2 is missing columns:{missing_cols}"

    print(f" Direction 1 feature count: {len(LSTM_FEATURES_V2)}")
    print("\n" + "=" * 60)
    print("【 Direction 1+3 】 LSTM")
    print("=" * 60)

    scaler_v2 = DataScaler(LSTM_FEATURES_V2)
    scaler_v2.fit(train_masked_v2)

    with open('scaler_v2.pkl', 'wb') as f:
        pickle.dump(scaler_v2, f)
    print("scaler_v2.pkl saved")

    model_v2 = train_lstm(train_masked_v2, train_orig_v2, scaler_v2,
                          window= 96, epochs=30, device=device)

    cut_i   = int(len(val_masked_v2) * 0.3)
    ft_end  = val_masked_v2.index[cut_i]
    eval_start = ft_end + pd.Timedelta(hours=1)

    val_ft_masked = val_masked_v2.loc[:ft_end]
    val_ft_orig   = val_orig_v2.loc[:ft_end]

    val_eval_masked = val_masked_v2.loc[eval_start:]
    val_eval_orig   = val_orig_v2.loc[eval_start:]
    val_eval_mask   = val_mask_v2.loc[eval_start:]

    model_v2_finetuned = finetune_on_target(
        model_v2, val_ft_masked, val_ft_orig, scaler_v2,
        window= 96, epochs=10, lr=1e-4, device=device)

    torch.save(model_v2_finetuned.state_dict(), 'model_v2_finetuned.pth')
    torch.save(model_v2.state_dict(), 'model_v2_base.pth')
    print("saved：model_v2_finetuned.pth, model_v2_base.pth")

    filled_train = impute_lstm(train_masked_v2, model_v2,
                               scaler_v2, window= 96, device=device)
    for r in evaluate_all(train_orig_v2, filled_train, train_mask_v2,
                          'LSTM', 'train'):
        all_results.append(r)

    filled_val_base = impute_lstm(val_eval_masked, model_v2,
                                  scaler_v2, window= 96, device=device)
    for r in evaluate_all(val_eval_orig, filled_val_base, val_eval_mask,
                          'LSTM', 'val30'):
        all_results.append(r)

    filled_val_ft = impute_lstm(val_eval_masked, model_v2_finetuned,
                                scaler_v2, window= 96, device=device)
    for r in evaluate_all(val_eval_orig, filled_val_ft, val_eval_mask,
                          'LSTM_FT', 'val30'):
        all_results.append(r)
        
    filled_test_base = impute_lstm(test_masked_v2, model_v2,
                                   scaler_v2, window= 96, device=device)
    for r in evaluate_all(test_orig_v2, filled_test_base, test_mask_v2,
                          'LSTM', 'test'):
        all_results.append(r)

    filled_test_ft = impute_lstm(test_masked_v2, model_v2_finetuned,
                                 scaler_v2, window= 96, device=device)
    for r in evaluate_all(test_orig_v2, filled_test_ft, test_mask_v2,
                          'LSTM_FT', 'test'):
        all_results.append(r)

    results_df = pd.DataFrame(all_results)
    results_df = results_df[['method', 'split', 'col', 'n_imputed', 'KGE', 'MAE', 'SMAPE(%)']]

    print("\n" + "=" * 60)
    for col in ['Q', 'v']:
        print(f"\n{'─' * 20} {col} {'─' * 20}")
        sub = (results_df[results_df['col'] == col]
               .sort_values(['split', 'KGE'], ascending=[True, False]))
        print(sub.to_string(index=False))
    print("\n" + "=" * 60)

    return results_df


device = 'cuda' if torch.cuda.is_available() else 'cpu'
comp_results = run_comparison(device=device)

Build feature dataset ..
 Direction 1 feature count: 55

【 Direction 1+3 】 LSTM
scaler_v2.pkl saved
    Epoch 10/30  Loss: 0.0026  LR: 1.00e-03
    Epoch 20/30  Loss: 0.0018  LR: 1.00e-03
    Epoch 30/30  Loss: 0.0013  LR: 1.00e-03
      FT Epoch 5/10  Loss: 0.000787  watch: 0.000503
      FT Epoch 10/10  Loss: 0.000683  watch: 0.000533
saved：model_v2_finetuned.pth, model_v2_base.pth


──────────────────── Q ────────────────────
 method split col  n_imputed    KGE    MAE  SMAPE(%)
LSTM_FT  test   Q        172 0.9734 0.0300    3.4746
   LSTM  test   Q        172 0.9529 0.0340    4.3920
   LSTM train   Q       1160 0.9884 0.0279    1.0729
LSTM_FT val30   Q         60 0.9576 0.0208    2.4677
   LSTM val30   Q         60 0.8981 0.0297    3.3279

──────────────────── v ────────────────────
 method split col  n_imputed    KGE    MAE  SMAPE(%)
LSTM_FT  test   v        139 0.9634 0.0066    2.0368
   LSTM  test   v        139 0.9413 0.0063    1.9559
   LSTM train   v        946 0.9954 0.0020   